#Criação da tabela Silver - MATCHES

## Objetivos
- Criação de tabela silver que capture dados da bronze sem ocorrência de duplicatas
- Filtragem apenas por partidas já encerradas
- Ajuste de datatypes de colunas
- Criação de colunas complementares

In [0]:
from pyspark.sql.functions import *

In [0]:
# coletando o máximo registro e transformando em uma váriavel para usar como filtro
max_dh_bronze = (
    # leitura da tabela bronze pra coleta da última data de atualização
    spark.read.table("api_football.bronze.matches_raw")
    # variável que captura a data mais recente de atualização (maior)
    .agg(max("dh_processing_br").alias("max_dh_bronze"))
    # transforma de dataframe para variável
    .collect()[0]["max_dh_bronze"]
)

# leitura da tabela de comparação, nesse caso a silver
df_silver = spark.read.table("api_football.silver.matches")

# join para pegar os dados que não estão na silver
df_matches_raw = (
    spark.read.table("api_football.bronze.matches_raw")
    .alias("mr")
    # left anti pega os dados das match_id que estão na bronze, mas não estão na silver
    .join(df_silver.alias("s"), col("s.match_id") == col("mr.match_id"), "leftanti")
    # filtra somente os dados que tem a data de atualização mais recente
    .filter(col("mr.dh_processing_br") == max_dh_bronze)
    # filtra somente as partidas finalizadas
    .filter(col("mr.match_status") == "Finished")
)

# display(df_matches_raw.count())

In [0]:
df_matches_clean = (
    # filtra por partidas finalizadas
    df_matches_raw.filter(col("match_status") == "Finished").select(
        # modifica datatype pra int devido ser coluna de valores númericos
        col("match_id").cast("int"),
        col("match_date"),
        col("match_hometeam_id").cast("int"),
        col("match_hometeam_name"),
        col("match_awayteam_id").cast("int"),
        col("match_awayteam_name"),
        col("match_hometeam_score").cast("int"),
        col("match_awayteam_score").cast("int"),
        # criação de coluna com o resultado da partida  -- scores iguais = draw
        when(col("match_hometeam_score") == col("match_awayteam_score"), "draw")
        # score do time da casa maior = home
        .when(
            col("match_hometeam_score").cast("int")
            > col("match_awayteam_score").cast("int"),
            "home",
        )
        # score do time visitante maior = away
        .when(
            col("match_hometeam_score").cast("int")
            < col("match_awayteam_score").cast("int"),
            "away",
        )
        # resulta na coluna match_result
        .alias("match_result"),
        col("match_round").cast("int"),
        col("match_stadium"),
        col("match_referee"),
        col("league_name"),
        # coluna com a data de processamento dos dados da bronze
        col("dh_processing_br").alias("dh_processing_bronze_br"),
        # coluna com a data de processamento do dataframe atual
        expr("current_timestamp()-interval 3 hours").alias("dh_processing_br"),
    )
    # ordenação
).orderBy(col("match_id").asc())

# display(df_matches_clean)

In [0]:
df_matches_clean.write.mode("append").saveAsTable("api_football.silver.matches")

In [0]:
%sql
select
  *
from
  api_football.silver.matches